In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.getOrCreate()

In [0]:
customers = spark.createDataFrame([
    (1,"John ","Hyderabad"),
    (2,"Alice","Chennai"),
    (3,None,"Bangalore")
],["customer_id","name","city"])

cars = spark.createDataFrame([
    (101,"Toyota","Camry",30000),
    (102,"Honda","Civic",-20000),
    (103,"Hyundai","i20",15000)
],["car_id","brand","model","price"])

sales = spark.createDataFrame([
    (1,1,101,"2024-01-01",1),
    (2,2,102,"2024-01-02",2),
    (3,99,103,"2024-01-03",1)
],["sale_id","customer_id","car_id","sale_date","quantity"])

In [0]:
sales = sales.withColumn("sale_date", to_date(col("sale_date")))

In [0]:
df = sales.join(customers,"customer_id").join(cars,"car_id")
df.groupBy("customer_id").sum("price").show()

+-----------+----------+
|customer_id|sum(price)|
+-----------+----------+
|          1|     30000|
|          2|    -20000|
+-----------+----------+



In [0]:
customers.display()

customer_id,name,city
1,John,Hyderabad
2,Alice,Chennai
3,null,Bangalore


In [0]:
cars.display()

car_id,brand,model,price
101,Toyota,Camry,30000
102,Honda,Civic,-20000
103,Hyundai,i20,15000


In [0]:
sales.display()

sale_id,customer_id,car_id,sale_date,quantity
1,1,101,2024-01-01,1
2,2,102,2024-01-02,2
3,99,103,2024-01-03,1


In [0]:
dealers = spark.createDataFrame([
    (1,"bob ","mumbai"),
    (2,"ram","hyderabad"),
    (3,"sam","vizag")
],["dealer_id","name","city"])

sales_dealer = spark.createDataFrame([
    (1,1),
    (2,3),
    (3,2)
],["saler_id","dealer_id"])

In [0]:
dealers.display()
sales_dealer.display()


dealer_id,name,city
1,bob,mumbai
2,ram,hyderabad
3,sam,vizag


saler_id,dealer_id
1,1
2,3
3,2


## print schema

In [0]:
customers.printSchema()
cars.printSchema()
sales.printSchema()
dealers.printSchema()
sales_dealer.printSchema()

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- car_id: long (nullable = true)
 |-- brand: string (nullable = true)
 |-- model: string (nullable = true)
 |-- price: long (nullable = true)

root
 |-- sale_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- car_id: long (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: long (nullable = true)

root
 |-- dealer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)

root
 |-- saler_id: long (nullable = true)
 |-- dealer_id: long (nullable = true)



### identify issues

1. Check Null Values

In [0]:
customers.select([sum(col(c).isNull().cast("int")).alias(c) for c in customers.columns]).show()
cars.select([sum(col(c).isNull().cast("int")).alias(c) for c in cars.columns]).show()
sales.select([sum(col(c).isNull().cast("int")).alias(c) for c in sales.columns]).show()
dealers.select([sum(col(c).isNull().cast("int")).alias(c) for c in dealers.columns]).show()
sales_dealer.select([sum(col(c).isNull().cast("int")).alias(c) for c in sales_dealer.columns]).show()

+-----------+----+----+
|customer_id|name|city|
+-----------+----+----+
|          0|   1|   0|
+-----------+----+----+

+------+-----+-----+-----+
|car_id|brand|model|price|
+------+-----+-----+-----+
|     0|    0|    0|    0|
+------+-----+-----+-----+

+-------+-----------+------+---------+--------+
|sale_id|customer_id|car_id|sale_date|quantity|
+-------+-----------+------+---------+--------+
|      0|          0|     0|        0|       0|
+-------+-----------+------+---------+--------+

+---------+----+----+
|dealer_id|name|city|
+---------+----+----+
|        0|   0|   0|
+---------+----+----+

+--------+---------+
|saler_id|dealer_id|
+--------+---------+
|       0|        0|
+--------+---------+



2. Check Duplicates

In [0]:
customers.groupBy(customers.columns).count().filter("count > 1").show()
cars.groupBy(cars.columns).count().filter("count > 1").show()
sales.groupBy(sales.columns).count().filter("count > 1").show()
dealers.groupBy(dealers.columns).count().filter("count > 1").show()
sales_dealer.groupBy(sales_dealer.columns).count().filter("count > 1").show()

+-----------+----+----+-----+
|customer_id|name|city|count|
+-----------+----+----+-----+
+-----------+----+----+-----+

+------+-----+-----+-----+-----+
|car_id|brand|model|price|count|
+------+-----+-----+-----+-----+
+------+-----+-----+-----+-----+

+-------+-----------+------+---------+--------+-----+
|sale_id|customer_id|car_id|sale_date|quantity|count|
+-------+-----------+------+---------+--------+-----+
+-------+-----------+------+---------+--------+-----+

+---------+----+----+-----+
|dealer_id|name|city|count|
+---------+----+----+-----+
+---------+----+----+-----+

+--------+---------+-----+
|saler_id|dealer_id|count|
+--------+---------+-----+
+--------+---------+-----+



3. Check Negative Prices

In [0]:
cars.filter(col("price") < 0).show()

+------+-----+-----+------+
|car_id|brand|model| price|
+------+-----+-----+------+
|   102|Honda|Civic|-20000|
+------+-----+-----+------+



4.Check Invalid Keys

In [0]:
sales.join(customers, "customer_id", "left_anti").show()

+-----------+-------+------+----------+--------+
|customer_id|sale_id|car_id| sale_date|quantity|
+-----------+-------+------+----------+--------+
|         99|      3|   103|2024-01-03|       1|
+-----------+-------+------+----------+--------+



# Phase 2 – Cleaning

### 1.Handle nulls

In [0]:

customers_clean = customers.fillna({
    "name": "Unknown",
    "city": "Unknown"
})

cars_clean = cars.fillna({
    "brand": "Unknown",
    "model": "Unknown",
    "price": 0
})

sales_clean = sales.fillna({
    "quantity": 1
})

###2. Fix Negative Price

In [0]:
cars_clean = cars_clean.withColumn("price", abs(col("price")))

### 3. Remove Invalid Keys

In [0]:
sales_clean = sales_clean.join(
    customers_clean,
    "customer_id",
    "inner"   
)
sales_clean.show()

+-----------+-------+------+----------+--------+-----+---------+-----+---------+-----+---------+
|customer_id|sale_id|car_id| sale_date|quantity| name|     city| name|     city| name|     city|
+-----------+-------+------+----------+--------+-----+---------+-----+---------+-----+---------+
|          1|      1|   101|2024-01-01|       1|John |Hyderabad|John |Hyderabad|John |Hyderabad|
|          2|      2|   102|2024-01-02|       2|Alice|  Chennai|Alice|  Chennai|Alice|  Chennai|
+-----------+-------+------+----------+--------+-----+---------+-----+---------+-----+---------+



# Phase 3 – Validation

### 1. Anti Join (Find Invalid Records)

A. Invalid customer_id in sales

In [0]:
invalid_customer_sales = sales_clean.join(
    customers_clean,
    "customer_id",
    "left_anti"
)

invalid_customer_sales.show()

+-----------+-------+------+---------+--------+----+----+
|customer_id|sale_id|car_id|sale_date|quantity|name|city|
+-----------+-------+------+---------+--------+----+----+
+-----------+-------+------+---------+--------+----+----+



B. Invalid car_id in sales

In [0]:
invalid_car_sales = sales_clean.join(
    cars_clean,
    "car_id",
    "left_anti"
)

invalid_car_sales.show()

+------+-----------+-------+---------+--------+----+----+
|car_id|customer_id|sale_id|sale_date|quantity|name|city|
+------+-----------+-------+---------+--------+----+----+
+------+-----------+-------+---------+--------+----+----+



C. Invalid dealer_id in sales_dealer

In [0]:
invalid_dealer_sales = sales_dealer.join(
    dealers,
    "dealer_id",
    "left_anti"
)

invalid_dealer_sales.show()

+---------+--------+
|dealer_id|saler_id|
+---------+--------+
+---------+--------+



### 2. Validation Report

A.Count invalid records

In [0]:
invalid_customer_count = invalid_customer_sales.count()
invalid_car_count = invalid_car_sales.count()
invalid_dealer_count = invalid_dealer_sales.count()


B.Total records


In [0]:
total_customers = customers_clean.count()
total_cars = cars_clean.count()
total_dealers = dealers.count()
total_sales = sales_clean.count()
total_sales_dealer = sales_dealer.count()

Create validation report

In [0]:
from pyspark.sql import Row

validation_data = [
    Row(check="Invalid Customer IDs", count=invalid_customer_count),
    Row(check="Invalid Car IDs", count=invalid_car_count),
    Row(check="Invalid Dealer IDs", count=invalid_dealer_count),
    Row(check="Total Sales Records", count=total_sales),
    Row(check="Total Sales Dealer Records", count=total_sales_dealer)
]

validation_report_df = spark.createDataFrame(validation_data)
validation_report_df.show()

+--------------------+-----+
|               check|count|
+--------------------+-----+
|Invalid Customer IDs|    0|
|     Invalid Car IDs|    0|
|  Invalid Dealer IDs|    0|
| Total Sales Records|    2|
|Total Sales Deale...|    3|
+--------------------+-----+



# Phase 4 – Transformation

### 1 Revenue per customer

In [0]:
sales_with_price = sales_clean.join(
    cars_clean,
    "car_id",
    "inner"
)
sales_with_revenue = sales_with_price.withColumn(
    "revenue",
    col("price") * col("quantity")
)
revenue_per_customer = sales_with_revenue.groupBy("customer_id") \
    .sum("revenue") \
    .withColumnRenamed("sum(revenue)", "total_revenue")

revenue_per_customer.show()

+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|          1|        30000|
|          2|        40000|
+-----------+-------------+



### 2. Cars per Brand

In [0]:
cars_per_brand = cars_clean.groupBy("brand") \
    .count() \
    .withColumnRenamed("count", "total_cars")

cars_per_brand.show()

+-------+----------+
|  brand|total_cars|
+-------+----------+
| Toyota|         1|
|  Honda|         1|
|Hyundai|         1|
+-------+----------+



### 3. City Revenue

In [0]:
from pyspark.sql.functions import col

# Step 1: Alias all DataFrames (to avoid ambiguous column error)
sales_alias = sales_clean.alias("s")
cars_alias = cars_clean.alias("c")
customers_alias = customers_clean.alias("cu")

# Step 2: Join tables
sales_full = sales_alias \
    .join(cars_alias, col("s.car_id") == col("c.car_id"), "inner") \
    .join(customers_alias, col("s.customer_id") == col("cu.customer_id"), "inner")

# Step 3: Calculate revenue
sales_full = sales_full.withColumn(
    "revenue",
    col("c.price") * col("s.quantity")
)

# Step 4: Aggregate city-wise revenue
city_revenue = sales_full.groupBy(col("cu.city").alias("city")) \
    .sum("revenue") \
    .withColumnRenamed("sum(revenue)", "total_revenue")

# Step 5: Show output
city_revenue.show(10, False)

+---------+-------------+
|city     |total_revenue|
+---------+-------------+
|Hyderabad|30000        |
|Chennai  |40000        |
+---------+-------------+



# Phase 5 – SQL Analysis

In [0]:
sales_full.createOrReplaceTempView("sales_full")
sales_clean.createOrReplaceTempView("sales")
customers_clean.createOrReplaceTempView("customers")
cars_clean.createOrReplaceTempView("cars")

### 1. Top 3 Customers per City

In [0]:
%sql
SELECT city, customer_id, total_revenue
FROM (
    SELECT 
        cu.city,
        s.customer_id,
        SUM(c.price * s.quantity) AS total_revenue,
        ROW_NUMBER() OVER (PARTITION BY cu.city ORDER BY SUM(c.price * s.quantity) DESC) AS rn
    FROM sales s
    JOIN cars c ON s.car_id = c.car_id
    JOIN customers cu ON s.customer_id = cu.customer_id
    GROUP BY cu.city, s.customer_id
) t
WHERE rn <= 3;

city,customer_id,total_revenue
Chennai,2,40000
Hyderabad,1,30000


### 2. Repeat Customers

In [0]:
%sql
SELECT 
    customer_id,
    COUNT(*) AS total_orders
FROM sales
GROUP BY customer_id
HAVING COUNT(*) > 1;

customer_id,total_orders


### 3. Monthly Trend

In [0]:
%sql
SELECT 
    DATE_FORMAT(sale_date, 'yyyy-MM') AS month,
    SUM(c.price * s.quantity) AS total_revenue
FROM sales s
JOIN cars c ON s.car_id = c.car_id
GROUP BY DATE_FORMAT(sale_date, 'yyyy-MM')
ORDER BY month;

month,total_revenue
2024-01,70000


### Phase 6 – Save Output

In [0]:
revenue_per_customer.write .mode("overwrite") .saveAsTable("revenue_per_customer_output")
cars_per_brand.write .mode("overwrite") .saveAsTable("cars_per_brand_output")
city_revenue.write .mode("overwrite") .saveAsTable("city_revenue_output")
dealers.write.mode("overwrite").saveAsTable("dealers_table")
sales_dealer.write.mode("overwrite").saveAsTable("sales_dealer_table")